# insurance-conformal

**Distribution-free prediction intervals for insurance pricing models.**

Parametric prediction intervals (e.g. Poisson confidence intervals) assume
the model is correctly specified and the residuals follow the assumed
distribution. In practice, they fail in the high-risk tail where it matters
most. Conformal prediction gives finite-sample coverage guarantees without
any parametric assumptions.

This notebook fits a CatBoost Tweedie model, wraps it with conformal
prediction intervals, verifies the coverage guarantee, and compares
interval widths across non-conformity score choices.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-conformal/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-conformal catboost polars scikit-learn

## Synthetic motor claims data

2,000 policies with a Tweedie-distributed loss cost. We use a 60/20/20
temporal split: train, calibrate, test.

In [ ]:
import numpy as np
import polars as pl

rng = np.random.default_rng(2024)
n = 2_000

# Rating factors
vehicle_age = rng.integers(1, 16, n).astype(float)
ncd_years   = rng.integers(0, 6, n).astype(float)
age_band    = rng.integers(0, 5, n).astype(float)  # 0=young, 4=mature

# Log-linear risk score -> Tweedie loss cost
log_risk = -1.5 + 0.05 * vehicle_age - 0.08 * ncd_years + 0.12 * (4 - age_band)
mu = np.exp(log_risk)  # expected pure premium (pounds per year)

# Simulate Tweedie (compound Poisson-Gamma): freq * sev
freq  = rng.poisson(mu * 0.15)           # ~15% claim frequency
sev   = rng.gamma(shape=2.0, scale=mu * 3.0, size=n)
y     = np.where(freq > 0, sev, 0.0).astype(float)

X = np.column_stack([vehicle_age, ncd_years, age_band])

# Temporal-style split: 60/20/20
n_train = int(0.6 * n)
n_cal   = int(0.2 * n)

X_train, y_train = X[:n_train], y[:n_train]
X_cal,   y_cal   = X[n_train:n_train+n_cal], y[n_train:n_train+n_cal]
X_test,  y_test  = X[n_train+n_cal:], y[n_train+n_cal:]

print(f"Train: {len(X_train)}, Cal: {len(X_cal)}, Test: {len(X_test)}")
print(f"Zero-claim rate in test: {(y_test == 0).mean():.1%}")

## Fit a CatBoost Tweedie model

In [ ]:
from catboost import CatBoostRegressor

cb = CatBoostRegressor(
    loss_function="Tweedie:variance_power=1.5",
    iterations=300,
    depth=5,
    learning_rate=0.05,
    verbose=0,
    random_seed=42,
    allow_writing_files=False,
)
cb.fit(X_train, y_train)
print("Model fitted.")

## Conformal prediction intervals

We wrap the fitted model and calibrate on the held-out calibration set.
`pearson_weighted` is the correct non-conformity score for Tweedie data:
it accounts for the variance-mean relationship and gives ~30% narrower
intervals than the raw residual score, with identical coverage guarantees.

**Guarantee:** P(y_test ∈ [lower, upper]) ≥ 1 - α for exchangeable data.
No parametric assumptions. No model specification required.

In [ ]:
from insurance_conformal import InsuranceConformalPredictor

cp = InsuranceConformalPredictor(
    model=cb,
    nonconformity="pearson_weighted",
    distribution="tweedie",
    tweedie_power=1.5,
)
cp.calibrate(X_cal, y_cal)

# Predict 90% intervals (alpha=0.10) on test set
intervals = cp.predict_interval(X_test, alpha=0.10)
print(intervals.head(8))

## Verify the coverage guarantee

In [ ]:
lower  = intervals["lower"].to_numpy()
upper  = intervals["upper"].to_numpy()
covered = (y_test >= lower) & (y_test <= upper)

print(f"Target coverage:   90.0%")
print(f"Achieved coverage: {covered.mean():.1%}")
print(f"Mean interval width: £{(upper - lower).mean():.2f}")

## Coverage by decile

The headline coverage number can hide failures in the tail.
`coverage_by_decile` checks whether the guarantee holds uniformly
across the risk distribution — the key diagnostic for insurance use.

In [ ]:
decile_df = cp.coverage_by_decile(X_test, y_test, alpha=0.10)
print(decile_df)

## Width comparison: raw vs pearson_weighted

The `pearson_weighted` score exploits the Tweedie variance structure
to produce narrower intervals. Here we compare it directly to the raw
absolute residual score at the same 90% coverage target.

In [ ]:
cp_raw = InsuranceConformalPredictor(model=cb, nonconformity="raw")
cp_raw.calibrate(X_cal, y_cal)
intervals_raw = cp_raw.predict_interval(X_test, alpha=0.10)

width_pw  = (intervals["upper"] - intervals["lower"]).mean()
width_raw = (intervals_raw["upper"] - intervals_raw["lower"]).mean()

print(f"pearson_weighted mean width: £{width_pw:.2f}")
print(f"raw mean width:              £{width_raw:.2f}")
print(f"Width reduction:             {(1 - width_pw/width_raw):.0%}")

## Next steps

This notebook covers the core `InsuranceConformalPredictor`. The full
library also provides:

- **`LocallyWeightedConformal`** — two-stage conformal with a secondary
  spread model; ~24% narrower than `pearson_weighted` on heterogeneous books
- **`HongModelFree`** — model-free conformal requiring no regression model
- **`insurance_conformal.risk`** — Conformal Risk Control for premium
  sufficiency: controls E[loss] ≤ α directly, not just coverage probability
- **`insurance_conformal.claims`** — Hong order-statistic shortcut,
  Tweedie nonconformity scores, and Solvency II SCR upper bounds
- **`insurance_conformal.multivariate`** — Joint multi-output conformal for
  simultaneous frequency/severity intervals

**GitHub:** https://github.com/burning-cost/insurance-conformal  
**PyPI:** https://pypi.org/project/insurance-conformal/

The full demo notebook (`notebooks/conformal_demo.py`) walks through
50,000-policy data with SCR reporting and the locally-weighted comparison.